# Experiment no 1

## 1. Aim
To apply basic feature engineering techniques, including handling missing values, encoding categorical variables, and scaling numerical attributes, on a suitable open-access dataset to prepare a clean and model-ready dataset for machine learning analysis.

## 2. Objectives
1. Understand the importance of feature engineering in machine learning.
2. Load and explore an open-access dataset using Python.
3. Identify and handle missing values in numerical and categorical attributes.
4. Convert categorical attributes into numerical form using one-hot encoding.
5. Scale numerical attributes using standardization.
6. Verify the processed dataset and prepare it for machine learning algorithms.

## 3. Software and Hardware Requirements
- **Programming language:** Python 3.x
- **Platform:** Google Colab / Jupyter Notebook
- **Libraries:** Pandas, NumPy, Scikit-learn
- **Hardware:** Computer with at least 4 GB RAM
- **Dataset:** Titanic passenger dataset (open access)

## 4. Theory
### 4.1 Feature Engineering
Feature engineering is the process of preparing, transforming, and selecting data attributes so that machine learning algorithms can use them effectively. Real-world datasets may contain missing values, categorical data, different numerical scales, and inconsistent information.

### 4.2 Handling Missing Values
- **Median imputation:** Replaces missing numerical values with the median; useful when data may contain outliers.
- **Most-frequent-value imputation:** Replaces missing categorical values with the most frequent category.
- Other possible methods include mean imputation and constant-value imputation.

### 4.3 Encoding Categorical Variables
Machine learning algorithms generally require numerical input. One-hot encoding creates separate binary columns for categories, making it suitable for nominal attributes such as sex and embarkation port.

### 4.4 Scaling Numerical Attributes
Standardization transforms a numerical feature using:
$$z = \frac{x - \mu}{\sigma}$$
where $x$ is the original value, $\mu$ is the training-set mean, and $\sigma$ is the training-set standard deviation. Standardization is useful for scale-sensitive algorithms such as Logistic Regression, SVM, and K-Nearest Neighbors.

## 5. Dataset Description
This experiment uses the Titanic passenger dataset from the public Seaborn-data repository.
- **Target:** `survived` (0 = did not survive, 1 = survived)
- **Selected input attributes:** `pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`, and `embarked`
- **Dataset URL:** `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv`

## 6. Procedure
1. Import the required Python libraries.
2. Load and explore the dataset.
3. Select relevant input features and target variable.
4. Split the data into training and testing sets.
5. Impute missing numerical and categorical values.
6. Apply one-hot encoding to categorical variables.
7. Standardize numerical variables.
8. Verify the processed data and save training/testing CSV files.


### Step 1: Import Required Libraries


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Libraries imported successfully.")


### Step 2: Load and Explore the Dataset


In [ ]:
# Load local dataset or fallback to remote repository
import os
import urllib.request

csv_path = "titanic.csv"
if not os.path.exists(csv_path):
    url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
    urllib.request.urlretrieve(url, csv_path)

df = pd.read_csv(csv_path)

print("First five records:")
display(df.head())

print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame(name="Data Type"))

print("\nMissing values in each column:")
display(df.isnull().sum().to_frame(name="Missing Values"))


### Step 3: Select Input Features and Target Variable


In [ ]:
features = [
    "pclass", "sex", "age",
    "sibsp", "parch", "fare", "embarked"
]

X = df[features].copy()
y = df["survived"].copy()

# Remove rows where the target is missing, if any
valid_rows = y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows].astype(int)

print("Input feature shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
print("\nTarget class counts:")
display(y.value_counts().sort_index())


### Step 4: Split the Dataset into Training and Testing Sets


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])
print("Training target distribution:")
display(y_train.value_counts(normalize=True).rename("Proportion"))
print("Testing target distribution:")
display(y_test.value_counts(normalize=True).rename("Proportion"))


### Step 5: Identify Numerical and Categorical Attributes


In [ ]:
numeric_features = ["age", "sibsp", "parch", "fare"]
categorical_features = ["pclass", "sex", "embarked"]

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

print("\nMissing values in training input:")
display(X_train.isnull().sum().to_frame(name="Missing Values"))


### Step 6: Handle Missing Values


In [ ]:
# Numerical attributes: median imputation
numeric_imputer = SimpleImputer(strategy="median")

# Categorical attributes: most-frequent-value imputation
categorical_imputer = SimpleImputer(strategy="most_frequent")

# Fit imputers on training data only
X_train_num_imputed = numeric_imputer.fit_transform(
    X_train[numeric_features]
)
X_test_num_imputed = numeric_imputer.transform(
    X_test[numeric_features]
)

X_train_cat_imputed = categorical_imputer.fit_transform(
    X_train[categorical_features]
)
X_test_cat_imputed = categorical_imputer.transform(
    X_test[categorical_features]
)

print("Missing values in imputed numerical training data:",
      np.isnan(X_train_num_imputed).sum())
print("Missing values in imputed numerical testing data:",
      np.isnan(X_test_num_imputed).sum())

print("\nNumerical medians learned from training data:")
display(pd.Series(
    numeric_imputer.statistics_,
    index=numeric_features,
    name="Training Median"
))

print("\nMost frequent categories learned from training data:")
display(pd.Series(
    categorical_imputer.statistics_,
    index=categorical_features,
    name="Training Mode"
))


### Step 7: Encode Categorical Variables Using One-Hot Encoding


In [ ]:
# Compatibility with both newer and older Scikit-learn versions
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

X_train_encoded = encoder.fit_transform(X_train_cat_imputed)
X_test_encoded = encoder.transform(X_test_cat_imputed)

encoded_feature_names = encoder.get_feature_names_out(
    categorical_features
)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded testing shape:", X_test_encoded.shape)

print("\nGenerated categorical feature names:")
print(encoded_feature_names)

display(pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names
).head())


### Step 8: Scale Numerical Attributes Using StandardScaler


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_num_imputed)
X_test_scaled = scaler.transform(X_test_num_imputed)

print("Scaled training numerical data:")
display(pd.DataFrame(
    X_train_scaled,
    columns=numeric_features
).head())

print("\nMeans of scaled training features (approximately 0):")
display(pd.Series(
    X_train_scaled.mean(axis=0).round(3),
    index=numeric_features,
    name="Mean"
))

print("\nStandard deviations of scaled training features (approximately 1):")
display(pd.Series(
    X_train_scaled.std(axis=0).round(3),
    index=numeric_features,
    name="Standard Deviation"
))


### Step 9: Build an Integrated Preprocessing Pipeline


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", one_hot_encoder)
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Learn preprocessing parameters from training data only
X_train_ready = preprocessor.fit_transform(X_train)

# Apply learned transformations to test data
X_test_ready = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(
    X_train_ready,
    columns=feature_names,
    index=X_train.index
)

X_test_df = pd.DataFrame(
    X_test_ready,
    columns=feature_names,
    index=X_test.index
)

print("Final training feature shape:", X_train_df.shape)
print("Final testing feature shape:", X_test_df.shape)
print("\nModel-ready training features:")
display(X_train_df.head())


### Step 10: Verify and Save the Model-Ready Datasets


In [ ]:
# Add target labels back to the processed features
train_ready = X_train_df.copy()
train_ready["survived"] = y_train

test_ready = X_test_df.copy()
test_ready["survived"] = y_test

print("Missing values in processed training dataset:",
      int(train_ready.isnull().sum().sum()))
print("Missing values in processed testing dataset:",
      int(test_ready.isnull().sum().sum()))

print("\nTraining dataset shape:", train_ready.shape)
print("Testing dataset shape:", test_ready.shape)

print("\nProcessed training data preview:")
display(train_ready.head())

# Save CSV files in the current notebook directory
train_ready.to_csv("titanic_train_model_ready.csv", index=False)
test_ready.to_csv("titanic_test_model_ready.csv", index=False)

print("\nFiles saved:")
print("- titanic_train_model_ready.csv")
print("- titanic_test_model_ready.csv")


## 7. Expected Observations
- Dataset loading: Commonly 891 records and 15 original columns.
- Feature selection: 7 input attributes.
- Train/test split: Approximately 80% training and 20% testing.
- Missing-value handling: Selected features have no missing values after imputation.
- Categorical encoding: Categories converted into binary columns.
- Numerical scaling: Numerical features standardized.
- Output: Two model-ready CSV files.

## 8. Result
The Titanic dataset was loaded and explored. Missing numerical and categorical values were handled using median and most-frequent-value imputation, respectively. Categorical variables were transformed using one-hot encoding, and numerical attributes were standardized using `StandardScaler`. The processed training and testing datasets were saved as CSV files.

## 9. Conclusion
Basic feature engineering techniques were applied to an open-access dataset using Python, Pandas, NumPy, and Scikit-learn. The experiment demonstrated how missing-value imputation, categorical encoding, and numerical scaling prepare raw data for machine learning analysis. Fitting preprocessing steps only on the training data reduces the risk of data leakage and supports reproducible model development.

## 10. Viva Voce Questions and Answers

**Q1. What is feature engineering?**  
*Answer:* Feature engineering is the process of preparing and transforming raw data into useful features for machine learning.

**Q2. Why are missing values handled?**  
*Answer:* Missing values may cause errors or reduce model effectiveness. Imputation provides usable values without discarding entire records.

**Q3. What is the difference between one-hot encoding and label encoding?**  
*Answer:* One-hot encoding creates binary columns for categories (avoiding spurious ordering), whereas label encoding assigns an arbitrary integer to each category.

**Q4. Why is StandardScaler used?**  
*Answer:* It standardizes numerical features to have zero mean and unit variance ($z = \frac{x - \mu}{\sigma}$), which is critical for distance and gradient-based algorithms.

**Q5. What is data leakage?**  
*Answer:* Data leakage occurs when information from the test set or future/unavailable data improperly influences model preprocessing or training.

**Q6. Why split data into training and testing sets?**  
*Answer:* Training data is used to learn parameters, while testing data evaluates model performance on unseen records to test true generalization.
